In [31]:
import sympy as sp
import numpy as np

# Initialize printing for clear math rendering in Jupyter/terminals
sp.init_printing(use_unicode=True)

# 1. Pathway & Physical Infrastructure Fill Ratios

## 1.1 Conduit Fill Ratio (NEC Chapter 9, Table 1 & BICSI TDMM)

Conduit fill limits prevent excessive pulling tension, jacket damage, and heat accumulation during cable installations. The National Electrical Code (NEC) specifies maximum cross-sectional area limits based on conductor count:

* 1 Cable: $53\%$ maximum fill
* 2 Cables: $31\%$ maximum fill (derated to prevent cable jamming when cables twist)
* 3 or more Cables: $40\%$ maximum fill

BICSI TDMM requirement

$$\text{Fill}_{\%} = \left( \frac{\sum_{i=1}^{n} \frac{\pi \cdot d_i^2}{4}}{\frac{\pi \cdot D_{\text{inner}}^2}{4}} \right) \times 100 = \left( \frac{A_{\text{cables}}}{A_{\text{conduit}}} \right) \times 100$$For $N$ identical cables of outer diameter $d$:$$\text{Fill}_{\%} = \left( \frac{N \cdot d^2}{D_{\text{inner}}^2} \right) \times 100$$

### Code

In [4]:
# Conduit Fill Symbols
d_cable, D_inner, A_cables_total, Fill_pct = sp.symbols(
    'd_cable D_inner A_cables_total Fill_pct', positive=True, real=True
)
N_cables = sp.symbols('N_cables', integer=True, positive=True)

# Equations
eq_conduit_area = sp.Eq(sp.Symbol('A_conduit'), sp.pi * (D_inner / 2)**2)
eq_identical_conduit_fill = sp.Eq(
    Fill_pct, 
    (N_cables * (sp.pi * (d_cable / 2)**2) / (sp.pi * (D_inner / 2)**2)) * 100
)
eq_general_conduit_fill = sp.Eq(
    Fill_pct, 
    (A_cables_total / (sp.pi * (D_inner / 2)**2)) * 100
)

# Analytical Solutions
# 1. Minimum conduit inner diameter required for a specified cable bundle area and fill target:
min_D_inner = sp.solve(eq_general_conduit_fill, D_inner)[0]

# 2. Maximum number of uniform cables allowable in a given conduit:
max_N_conduit_cables = sp.solve(eq_identical_conduit_fill, N_cables)[0]

eq_identical_conduit_fill

                               2
           100⋅N_cables⋅d_cable 
Fill_pct = ─────────────────────
                        2       
                  Dᵢₙₙₑᵣ        

### Examples

## 1.2 Cable Tray Fill (NEC 392 & TIA-569)

Cable tray cross-sectional fill evaluates volume constraints for ladder, ventilated trough, or wire mesh pathways. Design standards (such as ANSI/TIA-569) typically mandate a maximum initial planning fill of $40\%$ to accommodate pathway accessibility and future churn, with an absolute physical limit of $50\%$.$$\text{Tray Fill}_{\%} = \left( \frac{\sum_{k} N_k \cdot \left( \frac{\pi \cdot d_k^2}{4} \right)}{W_{\text{tray}} \cdot H_{\text{tray}}} \right) \times 100$$

### Code

In [6]:
# Cable Tray Fill Symbols
W_tray, H_tray, A_tray, Tray_Fill_pct = sp.symbols(
    'W_tray H_tray A_tray Tray_Fill_pct', positive=True, real=True
)

# Equations
eq_tray_area = sp.Eq(A_tray, W_tray * H_tray)
eq_tray_fill = sp.Eq(Tray_Fill_pct, (A_cables_total / (W_tray * H_tray)) * 100)

# Analytical Solution: Minimum tray width required given cable bundle area, depth, and max fill:
min_W_tray = sp.solve(eq_tray_fill, W_tray)[0]

eq_tray_fill

                100⋅A_cables_total
Tray_Fill_pct = ──────────────────
                  H_tray⋅W_tray   

### Examples

## 1.3 Cable Tray Linear Weight Loading
Physical pathway design requires verifying that total cable weight plus pathway self-weight does not exceed structural support and bracket ratings (such as NEMA Class 12C, 20B, or 20C ratings in pounds per linear foot).$$W_{\text{linear}} = w_{\text{tray}} + \sum_{k} N_k \cdot w_k$$

### Code

In [9]:
# Cable Tray Weight Loading Symbols
w_tray, w_cables_total, W_linear = sp.symbols(
    'w_tray w_cables_total W_linear', positive=True, real=True
)

eq_tray_weight = sp.Eq(W_linear, w_tray + w_cables_total)

# Analytical Solution: Allowable total cable weight capacity given tray structural limit:
allowable_w_cables = sp.solve(eq_tray_weight, w_cables_total)[0]

eq_tray_weight

Wₗᵢₙₑₐᵣ = w_cables_total + w_tray

# 2. Optical Fiber Transmission & Link Engineering

## 2.1 Optical Link Power Budget (TIA-568 / ITU-T G.652)

End-to-end passive insertion loss accounts for glass attenuation across link length $L$, mated connector pairs, fusion or mechanical splices, and engineering design headroom (safety margin).$$\text{Loss}_{\text{total}} = (\alpha_{\lambda} \cdot L) + (N_{\text{conn}} \cdot L_{\text{conn}}) + (N_{\text{splice}} \cdot L_{\text{splice}}) + M_{\text{safety}}$$

### Code

In [19]:
# Optical Loss Budget Symbols

L_fiber, alpha_lambda = sp.symbols('L_fiber alpha_lambda', positive=True, real=True)

N_conn, N_splice = sp.symbols('N_conn N_splice', integer=True, nonnegative=True)

L_conn, L_splice, M_safety, Loss_total = sp.symbols(
    'L_conn L_splice M_safety Loss_total', positive=True, real=True
)

eq_optical_loss = sp.Eq(
    Loss_total, 
    (alpha_lambda * L_fiber) + (N_conn * L_conn) + (N_splice * L_splice) + M_safety
)

# Analytical Solution: Maximum optical reach (L_fiber) given allowable link budget:
max_L_fiber = sp.solve(eq_optical_loss, L_fiber)[0]

eq_optical_loss

Lossₜₒₜₐₗ = L_conn⋅N_conn + L_fiber⋅α_λ + L_splice⋅N_splice + M_safety

## 2.2 Optical Power Margin Verification

A communications link is viable if the difference between minimum transmitter launch power ($P_{\text{TX,min}}$) and receiver sensitivity ($P_{\text{RX,sens}}$) is greater than or equal to total passive channel loss.$$\text{Budget}_{\text{available}} = P_{\text{TX,min}} - P_{\text{RX,sens}}$$$$\text{Margin}_{\text{excess}} = \text{Budget}_{\text{available}} - \text{Loss}_{\text{total}}$$$$\text{Margin}_{\text{excess}} \ge 0\text{ dB}$$

In [20]:
# Optical Power Margin Symbols
P_TX_min, P_RX_sens, Budget_avail, Margin_excess = sp.symbols(
    'P_TX_min P_RX_sens Budget_avail Margin_excess', real=True
)

eq_budget_avail = sp.Eq(Budget_avail, P_TX_min - P_RX_sens)
eq_margin_excess = sp.Eq(Margin_excess, Budget_avail - Loss_total)

# Analytical Solution: Minimum required transmitter output to close the link:
min_P_TX = sp.solve(eq_margin_excess.subs(Budget_avail, P_TX_min - P_RX_sens), P_TX_min)[0]

eq_margin_excess

Margin_excess = Budgetₐᵥₐᵢₗ - Lossₜₒₜₐₗ

## 2.3 Chromatic Dispersion (CD) in Single-Mode Fiber

Pulse spreading resulting from the variance of glass refractive index across wavelength components ($\Delta\lambda$) of the optical transmitter over length $L$.$$\Delta t_{\text{CD}} = \vert{}D(\lambda)\vert{} \cdot \Delta\lambda \cdot L$$

In [21]:
# Chromatic Dispersion Symbols
D_lambda, delta_lambda, delta_t_CD = sp.symbols(
    'D_lambda delta_lambda delta_t_CD', positive=True, real=True
)

eq_chromatic_dispersion = sp.Eq(delta_t_CD, sp.Abs(D_lambda) * delta_lambda * L_fiber)

# Analytical Solution: Maximum uncompensated distance before exceeding dispersion threshold:
max_dispersion_length = sp.solve(eq_chromatic_dispersion, L_fiber)[0]

eq_chromatic_dispersion

δ_t_CD = D_λ⋅L_fiber⋅δ_λ

2.4 Polarization Mode Dispersion (PMD)

Differential group delay between orthogonal polarization modes caused by glass core non-circularity and asymmetric external mechanical stresses. PMD accumulates statistically with the square root of distance.$$\langle \Delta \tau_{\text{PMD}} \rangle = D_{\text{PMD}} \cdot \sqrt{L}$$

In [22]:
# PMD Symbols
D_PMD, delta_tau_PMD = sp.symbols('D_PMD delta_tau_PMD', positive=True, real=True)

eq_pmd = sp.Eq(delta_tau_PMD, D_PMD * sp.sqrt(L_fiber))

# Analytical Solution: Fiber distance threshold based on receiver PMD tolerance:
max_pmd_length = sp.solve(eq_pmd, L_fiber)[0]

eq_pmd

                  _________
δ_τ_PMD = D_PMD⋅╲╱ L_fiber 

# 3. Outside Plant (OSP) Cable Pulling Mechanics

## 3.1 Straight Horizontal Pulling Tension

Tension accumulation along a straight horizontal raceway driven by kinetic friction between the cable jacket and the inner conduit wall.$$T_{\text{out}} = T_{\text{in}} + (\mu \cdot w \cdot L)$$

In [23]:
# Straight Pulling Tension Symbols
T_in, T_out, mu, w_cable, L_straight = sp.symbols(
    'T_in T_out mu w_cable L_straight', positive=True, real=True
)
T_max = sp.symbols('T_max', positive=True, real=True)

eq_straight_pull = sp.Eq(T_out, T_in + (mu * w_cable * L_straight))

# Analytical Solution: Maximum straight run distance allowable before exceeding tensile limit:
max_pull_length = sp.solve(eq_straight_pull.subs(T_out, T_max), L_straight)[0]

eq_straight_pull

Tₒᵤₜ = L_straight⋅μ⋅w_cable + Tᵢₙ

## 3.2 Curved Duct Pulling Tension (Capstan Equation)

Tension amplification through horizontal or vertical conduit bends. Tension multiplies exponentially as a function of the bend angle $\theta$ (in radians) and the coefficient of friction $\mu$.$$T_{\text{out}} = T_{\text{in}} \cdot e^{\mu \cdot \theta}$$

In [24]:
# Capstan Equation Symbols
theta = sp.symbols('theta', positive=True, real=True) # Angle in radians

eq_capstan = sp.Eq(T_out, T_in * sp.exp(mu * theta))

# Analytical Solution: Maximum allowable entry tension entering a bend given exit limit:
allowable_T_in = sp.solve(eq_capstan, T_in)[0]

eq_capstan

            μ⋅θ
Tₒᵤₜ = Tᵢₙ⋅ℯ   

## 3.3 Sidewall Bearing Pressure (SWBP)

The radial crushing force exerted on the cable jacket and internal buffer tubes/conductors as the cable is pulled around a bend under tension. Exceeding SWBP ratings deforms optical geometry and induces microbending loss.$$\text{SWBP} = \frac{T_{\text{out}}}{R}$$

In [25]:
# Sidewall Bearing Pressure Symbols
R_bend, SWBP, SWBP_max = sp.symbols('R_bend SWBP SWBP_max', positive=True, real=True)

eq_swbp = sp.Eq(SWBP, T_out / R_bend)

# Analytical Solution: Minimum bend radius required to maintain SWBP within limits:
min_bend_radius = sp.solve(eq_swbp.subs(SWBP, SWBP_max), R_bend)[0]

eq_swbp

        Tₒᵤₜ 
SWBP = ──────
       R_bend

# 4. Copper Balanced Twisted-Pair & Transmission Lines
## 4.1 Insertion Loss Temperature De-Rating (ANSI/TIA-568.2-D)

Elevated ambient temperatures in ceiling plenums, risers, or dense cable bundles increase conductor resistance and dielectric loss, increasing channel insertion loss relative to standard reference conditions ($20^\circ\text{C}$).$$\text{IL}_T = \text{IL}_{20} \cdot \left[1 + c_T \cdot (T_{\text{amb}} - 20)\right]$$

In [26]:
# Copper Temperature De-rating Symbols
IL_20, IL_T, T_amb, c_T, IL_limit = sp.symbols(
    'IL_20 IL_T T_amb c_T IL_limit', positive=True, real=True
)

eq_copper_temp_derating = sp.Eq(IL_T, IL_20 * (1 + c_T * (T_amb - 20)))

# Analytical Solution: Maximum permissible ambient temperature before exceeding IL budget:
max_ambient_temp = sp.solve(eq_copper_temp_derating.subs(IL_T, IL_limit), T_amb)[0]

eq_copper_temp_derating

IL_T = IL₂₀⋅(c_T⋅(T_amb - 20) + 1)

## 4.2 Decibel Conversions & Return Loss

Fundamental conversions linking linear optical/RF power to logarithmic units and quantifying reflective impedance discontinuities.$$P_{(\text{dBm})} = 10 \cdot \log_{10}\left(\frac{P_{(\text{mW})}}{1\,\text{mW}}\right)$$$$P_{(\text{mW})} = 10^{\frac{P_{(\text{dBm})}}{10}}$$$$\text{RL} = -10 \cdot \log_{10}\left(\frac{P_{\text{refl}}}{P_{\text{inc}}}\right)$$

In [27]:
# Decibel and Return Loss Symbols
P_mW, P_dBm = sp.symbols('P_mW P_dBm', real=True)
P_refl, P_inc, RL = sp.symbols('P_refl P_inc RL', positive=True, real=True)

eq_mw_to_dbm = sp.Eq(P_dBm, 10 * sp.log(P_mW, 10))
eq_dbm_to_mw = sp.Eq(P_mW, 10**(P_dBm / 10))
eq_return_loss = sp.Eq(RL, -10 * sp.log(P_refl / P_inc, 10))

# Analytical Solution: Reflected power ratio as a function of Return Loss:
refl_ratio = sp.solve(eq_return_loss, P_refl / P_inc)[0]

eq_return_loss

            ⎛P_refl⎞ 
     -10⋅log⎜──────⎟ 
            ⎝P_inc ⎠ 
RL = ────────────────
         log(10)     

# 5. Data Center ICT Spaces & Environmental Loads
## 5.1 Heat Dissipation from ICT Equipment
All active computing, switching, and routing electrical energy is rejected as heat under steady-state conditions. Sensible cooling requirements are derived directly from electrical load:$$\text{Heat Load} (\text{BTU/hr}) = P_{\text{IT}} (\text{kW}) \times 3412.142$$$$\text{Cooling Tons} = \frac{\text{Heat Load} (\text{BTU/hr})}{12000} = \frac{P_{\text{IT}} \times 3412.142}{12000}$$

In [28]:
# Data Center Heat Load Symbols
P_IT_kW, Heat_BTU_hr, Cooling_Tons = sp.symbols(
    'P_IT_kW Heat_BTU_hr Cooling_Tons', positive=True, real=True
)

eq_heat_btu = sp.Eq(Heat_BTU_hr, P_IT_kW * sp.Rational(3412142, 1000))
eq_cooling_tons = sp.Eq(Cooling_Tons, Heat_BTU_hr / 12000)

eq_cooling_tons

               Heat_BTU_hr
Cooling_Tons = ───────────
                  12000   

## 5.2 Power Usage Effectiveness (PUE)

Quantifies the overhead energy expended on data center cooling, power delivery, lighting, and building systems relative to the direct energy consumed by IT hardware.$$\text{PUE} = \frac{P_{\text{total\_facility}}}{P_{\text{IT\_equipment}}}$$$$\text{Overhead Ratio} = \text{PUE} - 1.0$$

In [29]:
# PUE Symbols
P_facility, PUE = sp.symbols('P_facility PUE', positive=True, real=True)

eq_pue = sp.Eq(PUE, P_facility / P_IT_kW)

# Analytical Solution: Maximum total facility power permissible for a targeted PUE:
max_P_facility = sp.solve(eq_pue, P_facility)[0]

eq_pue

      P_facility
PUE = ──────────
       P_IT_kW  

In [32]:
# 1. Exact Symbolic Substitution Example (Optical Loss Budget)
exact_loss = eq_optical_loss.rhs.subs({
    alpha_lambda: 0.35,  # dB/km (SMF-28 @ 1310nm)
    L_fiber: 14.0,       # km
    N_conn: 4,           # 2 patch panels (4 mated pairs)
    L_conn: 0.50,        # dB per connector pair
    N_splice: 2,         # 2 fusion splices
    L_splice: 0.05,      # dB per splice
    M_safety: 1.5        # dB engineering margin
})
print(f"Exact Calculated Loss: {float(exact_loss):.3f} dB")

# 2. Vectorized NumPy Function Generation via lambdify (Capstan Bend Tension)
py_capstan = sp.lambdify(
    (T_in, mu, theta), 
    eq_capstan.rhs, 
    modules='numpy'
)

# Pulling around 90-degree bend (pi/2 rad) with 0.35 friction coefficient
test_t_in = np.array([100.0, 200.0, 300.0])  # lbf
test_t_out = py_capstan(test_t_in, 0.35, np.pi / 2)
print("Vectorized Capstan Exit Tensions (lbf):", np.round(test_t_out, 2))

Exact Calculated Loss: 8.500 dB
Vectorized Capstan Exit Tensions (lbf): [173.29 346.57 519.86]
